# Eval Server Submission Quickstart

This notebook walks through writing and submitting a featurizer to the evaluation server. The instructions here assume you are running it locally (on your computer), NOT in colab. It may be possible to run in Colab, but the google cloud setup might work differently and we have not tested this.

The server runs your `Featurizer` class against real CDR data from Togo (2018),
evaluates how well your features predict household consumption using nested cross-validation, and logs results to the leaderboard.

Models: Lasso, Ridge, Random Forest, Neural Net

**Metrics are reported for two runs:**
- `results_user_features_only` — your features alone
- `results_merged_with_existing` — your features merged with the existing CIDER feature set

## Prerequisites

### 1. Google Cloud / gcloud CLI
You need the `gcloud` CLI installed and authenticated. Please make sure to use your .berkeley.edu account; it will be in a special google group which gives it access to this API. Other accounts won't work.
```bash
gcloud auth login
gcloud config set project gol-cdr-featurization-comp
```


### 2. Python dependencies
Install `requests` package using Pip or Conda.

## Step 1 — Write your Featurizer
Create a .py file containing a featurizer class. You can do this manually, or by editing the following cell, which has a "magic" annotation (starting with %%) that causes its contents to be written to file. 

Your class must:
- Be named **`Featurizer`**
- Have a `name(self)` method returning a short string identifier
- Have a `featurize(self, cdr, mobile_money, mobile_data, recharges, antennas, shapefiles)` method
  that returns a **`pandas.DataFrame`** indexed by `phone_number`

**Available inputs** (all are pandas DataFrames / dicts of GeoDataFrames. For now `recharges` will be `None` since we are only using Togo data. Column names match the synthetic data.)

In [22]:
%%writefile featurizer_submission.py

#------ YOUR FEATURIZER HERE ---------- 
# This is just an example: Counts the number of total transactions the user has
import pandas as pd

class Featurizer:

    def name(self):
        return 'count transactions'

    def featurize(
        self,
        cdr=None,
        mobile_money=None,
        mobile_data=None,
        recharges=None,
        antennas=None,
        shapefiles=None
    ):
        outgoing = cdr.copy()
        outgoing.rename(
            columns={
                'caller_msisdn': 'ego',
                'recipient_msisdn': 'alter',
            },
            inplace=True
        )
        outgoing['direction'] = 'outgoing'

        incoming = cdr.copy()
        incoming.rename(
            columns={
                'caller_msisdn': 'alter',
                'recipient_msisdn': 'ego',
            },
            inplace=True
        )
        incoming['direction'] = 'incoming'

        bidirectional = pd.concat((outgoing, incoming))

        num_transactions = bidirectional.groupby('ego').apply(len).rename('num_transactions')
        return num_transactions.to_frame()

Overwriting featurizer_submission.py


## Step 2 — Submit to the evaluation server

This cell:
1. Looks up the Cloud Run service URL via `gcloud`
2. Gets your authenticated Google identity
3. Fetches a short-lived OIDC identity token (required since the service uses `--no-allow-unauthenticated`)
4. POSTs your featurizer code to the `/execute` endpoint

The server will run your featurizer, evaluate it, log results to the sheet, and return metrics.
Expect it to take a few minutes. It expects your featurizer to be in a file called featurizer_submission.py; you can edit that line if you'd like to name your file differently. 

In [29]:
import subprocess
import requests

SERVICE_URL = ' https://featurization-test-server-594466465768.us-central1.run.app'

# Your Google account email (used to identify your submission in the log/leaderboard). See prerequisites
# above for the steps to setting up your account in the command line.
USER_EMAIL = subprocess.run(
    ['gcloud', 'config', 'get-value', 'account'],
    capture_output=True, text=True, check=True
).stdout.strip()

id_token = subprocess.run(
    ['gcloud', 'auth', 'print-identity-token'],
    capture_output=True, text=True, check=True
).stdout.strip()

# Read the featurizer
with open('featurizer_submission.py', 'r') as f:
    code = f.read()

print(f"Service URL : {SERVICE_URL}")
print(f"Submitting as: {USER_EMAIL}")
print("Sending request...")

# The full_run argument determines whether random forest and neural network models are run. For now, full_run must be false.
response = requests.post(
    f"{SERVICE_URL}/execute",
    headers={"Authorization": f"Bearer {id_token}"},
    json={"code": code, 'user': USER_EMAIL, 'full_run': 'False'},
    timeout=60,
)

print(f"Status: {response.status_code}")

print(response.json())

Service URL :  https://featurization-test-server-594466465768.us-central1.run.app
Submitting as: selker@berkeley.edu
Sending request...
Status: 504


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [24]:
response.__dict__

{'_content': b'upstream request timeout',
 '_content_consumed': True,
 '_next': None,
 'status_code': 504,
 'headers': {'content-type': 'text/plain', 'x-cloud-trace-context': '7cda30ca5ab24e749af1b16ec3d9831d;o=1', 'date': 'Tue, 17 Mar 2026 23:18:26 GMT', 'server': 'Google Frontend', 'Content-Length': '24', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'},
 'raw': <urllib3.response.HTTPResponse at 0x1160a6490>,
 'url': 'https://featurization-test-server-594466465768.us-central1.run.app/execute',
 'encoding': 'ISO-8859-1',
 'history': [],
 'reason': 'Gateway Timeout',
 'cookies': <RequestsCookieJar[]>,
 'elapsed': datetime.timedelta(seconds=50, microseconds=300447),
 'request': <PreparedRequest [POST]>,
 'connection': <requests.adapters.HTTPAdapter at 0x115f0f340>}

## View results

All runs are automatically logged in [this](https://docs.google.com/spreadsheets/d/13vZkBNoI1TNKEuWJhtFospr_XNR23DpZTobjaKAJneA/edit?gid=0#gid=0) sheet. The first page logs all runs. The second and third are leaderboards, one for featurizers on their own and one for featurizers when their outputs are combined with our existing set of features. 